<a href="https://colab.research.google.com/github/Poojarautela03/ABTALKS/blob/main/Day26_Introduction%20to%20AI%20Agents%20and%20the%20ReAct%20Pattern/ReAct_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 26 — Introduction to AI Agents and the ReAct Pattern
**ABTalks 60-Day AI Challenge · Focus Area: AI Agents**

A standard LLM call answers a question in one step. An **agent** plans, uses tools, observes
results, and decides what to do next — repeatedly, autonomously. The only way to really
understand where that autonomy is solid and where it quietly breaks is to build the loop by
hand, with no framework (no LangChain agent, no AutoGPT) hiding the mechanics.

This notebook implements the **ReAct pattern** (Reason + Act) manually:

```
Thought  → the model reasons about what it needs next
Action   → it names a tool and an input
Observation → the tool actually runs, and its real output is fed back in
...repeat...
Final Answer → the model has enough to answer
```

**What this notebook does:**
1. Implements the full ReAct loop manually — no agent framework
2. Builds three tools: `calculator()`, `search_docs()` (against a Day 11-style knowledge base),
   and `get_today()`
3. Wires them into a `tool_registry` dict and a `dispatch()` function
4. Runs the agent on 5 multi-step problems that each require more than one tool
5. Prints the full Thought → Action → Observation trace for every step
6. **Automatically detects** (not just narrates) three real failure modes: looping without
   progress, hallucinating a result the tools never returned, and picking the wrong tool
7. Closes with a written reliability analysis grounded in what was actually observed

**Note on the LLM used here:** as in Day 24 and Day 25, this runs on a deterministic offline
mock (see `llm.py`) so the notebook is reproducible without an API key — set `OPENAI_API_KEY`
to swap in real OpenAI calls with no change to the loop itself (`agent.py`), the tools, or the
dispatcher. Unlike a simple keyword-matcher, this mock genuinely reads the real Observations
returned by the real tools (via regex) to decide its next step — and two of the five problems
have a **deliberately scripted failure**, because a mock that always reasons perfectly would
never produce the failure patterns this task asks us to find.


## 1. The three tools

Each tool is a plain `(str) -> str` function — the shape an LLM's Action line naturally
produces (a tool name plus one string argument) and the shape the dispatcher expects.

- **`calculator(expression)`** — safely evaluates arithmetic using Python's `ast` module
  (no `eval()`, no builtins reachable — only `+ - * / **` over numeric literals). Returns an
  `"ERROR: ..."` string instead of raising, so a bad expression becomes a normal Observation
  the agent can see and recover from.
- **`search_docs(query)`** — bag-of-words cosine similarity retrieval over a small Day
  11-style knowledge base (five facts with concrete numbers and dates, recreated here since
  Day 11 isn't in this notebook's history).
- **`get_today()`** — returns today's real date via `datetime.date.today()`.


In [1]:
%%writefile tools.py
"""
tools.py
--------
The three tools available to the agent. Each one is a plain Python function
that takes a single string argument (the way the LLM will pass it in an
Action line) and returns a string observation.
"""

from __future__ import annotations

import ast
import datetime
import math
import operator
import re
from collections import Counter
from typing import Dict, List

# ---------------------------------------------------------------------------
# Day 11 knowledge base (recreated here since Day 11 isn't in this notebook's
# history -- same approach as Day 25's Day 20 fixtures: a small, realistic
# corpus with concrete facts and numbers the agent's tools can actually work
# with).
# ---------------------------------------------------------------------------
KNOWLEDGE_BASE: List[Dict[str, str]] = [
    {"source": "doc_python", "text": "Python was first released by Guido van Rossum in 1991."},
    {"source": "doc_eiffel_tower", "text": "The Eiffel Tower is 330 meters tall and was completed in 1889."},
    {"source": "doc_everest", "text": "Mount Everest is 8849 meters tall, the highest mountain on Earth."},
    {"source": "doc_amazon", "text": "The Amazon rainforest covers approximately 5500000 square kilometers."},
    {"source": "doc_moon_landing", "text": "The Apollo 11 moon landing occurred on July 20, 1969."},
]


# --- Tool 1: calculator ----------------------------------------------------
_ALLOWED_OPS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.Pow: operator.pow,
    ast.USub: operator.neg,
    ast.UAdd: operator.pos,
}


def _safe_eval(node):
    """Recursively evaluate an arithmetic-only AST node (no names, no calls)."""
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _ALLOWED_OPS:
        return _ALLOWED_OPS[type(node.op)](_safe_eval(node.left), _safe_eval(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _ALLOWED_OPS:
        return _ALLOWED_OPS[type(node.op)](_safe_eval(node.operand))
    raise ValueError(f"Unsupported expression element: {ast.dump(node)}")


def calculator(expression: str) -> str:
    """
    Evaluate an arithmetic expression safely (no eval(), no builtins --
    only +, -, *, /, **, and parentheses over numeric literals).

    Args:
        expression: An arithmetic expression, e.g. "2026 - 1991" or "330 * 3.281".

    Returns:
        The result as a string, or an error message string starting with
        "ERROR:" if the expression isn't valid arithmetic (deliberately
        returned as a string, not raised, so the agent loop can feed the
        error back to the LLM as an Observation instead of crashing).
    """
    try:
        tree = ast.parse(expression, mode="eval")
        result = _safe_eval(tree.body)
        return str(result)
    except Exception as e:
        return f"ERROR: could not evaluate '{expression}' as arithmetic ({e})"


# --- Tool 2: search_docs ----------------------------------------------------
def _embed(text: str) -> Counter:
    words = re.findall(r"[a-z0-9']+", text.lower())
    return Counter(words)


def _cosine_similarity(vec_a: Counter, vec_b: Counter) -> float:
    common = set(vec_a) & set(vec_b)
    dot = sum(vec_a[w] * vec_b[w] for w in common)
    mag_a = math.sqrt(sum(v * v for v in vec_a.values()))
    mag_b = math.sqrt(sum(v * v for v in vec_b.values()))
    if mag_a == 0 or mag_b == 0:
        return 0.0
    return dot / (mag_a * mag_b)


def search_docs(query: str) -> str:
    """
    Retrieve the single most relevant snippet from the Day 11 knowledge base.

    Args:
        query: A free-text search query.

    Returns:
        The best-matching document, formatted as "[source] text", or a
        "No relevant documents found." message if nothing scores above zero.
    """
    query_vec = _embed(query)
    scored = [
        (_cosine_similarity(query_vec, _embed(doc["text"])), doc)
        for doc in KNOWLEDGE_BASE
    ]
    scored.sort(key=lambda x: x[0], reverse=True)
    best_score, best_doc = scored[0]
    if best_score == 0:
        return "No relevant documents found."
    return f"[{best_doc['source']}] {best_doc['text']}"


# --- Tool 3: get_today -------------------------------------------------------
def get_today(_: str = "") -> str:
    """
    Return today's date as a string.

    Args:
        _: Unused. Present so every tool shares the same (str) -> str
           signature and can be called uniformly by the dispatcher.

    Returns:
        Today's date in YYYY-MM-DD format.
    """
    return datetime.date.today().isoformat()

Writing tools.py


In [2]:
# Sanity check: each tool works in isolation
from tools import calculator, search_docs, get_today

print(calculator("330 * 3.281"))
print(calculator("2 +"))              # deliberately invalid, to see the error path
print(search_docs("Eiffel Tower height"))
print(search_docs("quantum computing"))  # deliberately no match
print(get_today())

1082.73
ERROR: could not evaluate '2 +' as arithmetic (invalid syntax (<unknown>, line 1))
[doc_eiffel_tower] The Eiffel Tower is 330 meters tall and was completed in 1889.
No relevant documents found.
2026-08-28


## 2. Tool registry and dispatch

`tool_registry` maps the tool name the LLM will write in an Action line to the actual Python
function. `dispatch()` looks up the name and calls it — and returns an `"ERROR: unknown
tool"` string (not a crash) if the LLM names something that isn't registered, since naming a
nonexistent tool is itself one of the failure modes an agent can produce.


In [3]:
%%writefile registry.py
"""
registry.py
-----------
Maps tool names (as the LLM will name them in an Action line) to the actual
Python functions, and dispatches a parsed Action to the right one.
"""

from __future__ import annotations

from typing import Callable, Dict

from tools import calculator, search_docs, get_today

tool_registry: Dict[str, Callable[[str], str]] = {
    "calculator": calculator,
    "search_docs": search_docs,
    "get_today": get_today,
}


def dispatch(action_name: str, action_input: str) -> str:
    """
    Route a parsed (action_name, action_input) pair to the matching tool.

    Args:
        action_name: The tool name the LLM named in its Action line.
        action_input: The raw string argument to pass to that tool.

    Returns:
        The tool's string result, or an "ERROR: unknown tool" message if
        action_name isn't in tool_registry (returned as a string, not
        raised, so this failure becomes a normal Observation the LLM can
        see and recover from -- this is itself one of the failure modes
        the task asks us to watch for: an agent naming a tool that doesn't
        exist).
    """
    if action_name not in tool_registry:
        available = ", ".join(tool_registry)
        return f"ERROR: unknown tool '{action_name}'. Available tools: {available}"
    tool_fn = tool_registry[action_name]
    return tool_fn(action_input)

Writing registry.py


## 3. The "LLM": five scripted problems

Each of the five problems below is implemented as a small step-function that reads the
**real** Observations produced so far and decides the next Thought/Action by extracting the
numbers/dates it needs — genuinely responsive reasoning, not a fixed transcript. Two problems
have an intentionally scripted failure (see the inline comments in `llm.py`), because
observing real failure patterns requires an agent that's capable of actually failing.


In [4]:
%%writefile llm.py
"""
llm.py
------
The "LLM" the agent calls at every step. Like Day 24's and Day 25's mocks,
this runs fully offline and deterministically so the notebook is
reproducible without an API key -- set OPENAI_API_KEY and swap call_llm()
for a real Chat Completions call and nothing else in the agent loop changes.

Unlike a simple keyword-matcher, this mock actually reads the real
Observations produced by the real tools (via regex) to decide its next
Thought/Action -- so the reasoning chain is genuinely responsive to what the
tools returned, not just replaying a fixed script blindly. Two of the five
problems below have a deliberately scripted failure mode (see PROBLEMS),
since the task asks us to observe and document specific failure patterns,
and a mock LLM that always reasons perfectly wouldn't produce any.
"""

from __future__ import annotations

import re
from typing import Callable, Dict, List, Optional


def _last_number(text: str) -> Optional[float]:
    """Pull the last integer/float literal out of a string, if any."""
    matches = re.findall(r"-?\d+\.?\d*", text)
    return float(matches[-1]) if matches else None


def _first_number(text: str) -> Optional[float]:
    """Pull the first integer/float literal out of a string, if any."""
    matches = re.findall(r"-?\d+\.?\d*", text)
    return float(matches[0]) if matches else None


# ---------------------------------------------------------------------------
# One step-function per problem. Each takes the Observations seen so far
# (already produced by the *real* tools) and returns the next step as a
# dict: either {"thought", "action", "input"} or {"thought", "final_answer"}.
# ---------------------------------------------------------------------------

def _p1_python_release(obs: List[str], step: int) -> Dict[str, str]:
    if step == 0:
        return {"thought": "I need to find out when Python was first released.",
                "action": "search_docs", "input": "Python first released year"}
    if step == 1:
        year = int(_last_number(obs[-1]))
        return {"thought": f"Python was released in {year}. I need today's date to compute how many years ago that was.",
                "action": "get_today", "input": ""}
    if step == 2:
        today = obs[-1]
        current_year = int(today[:4])
        release_year = int(_last_number(obs[-2]))
        return {"thought": f"Today is {today}, so I can calculate {current_year} - {release_year}.",
                "action": "calculator", "input": f"{current_year} - {release_year}"}
    years_ago = int(_last_number(obs[-1]))
    return {"thought": "I now have the number of years.",
            "final_answer": f"Python was first released in 1991, which is {years_ago} years ago (as of today)."}


def _p2_eiffel_feet(obs: List[str], step: int) -> Dict[str, str]:
    if step == 0:
        return {"thought": "I need the Eiffel Tower's height in meters before I can convert it.",
                "action": "search_docs", "input": "Eiffel Tower height meters"}
    if step == 1:
        meters = _first_number(obs[-1])  # doc text: "330 meters ... completed in 1889" -- height is the first number
        return {"thought": f"The tower is {meters:.0f} meters tall. 1 meter = 3.281 feet, so I'll multiply.",
                "action": "calculator", "input": f"{meters:.0f} * 3.281"}
    feet = _last_number(obs[-1])
    return {"thought": "I have the converted height.",
            "final_answer": f"The Eiffel Tower is approximately {feet:.1f} feet tall."}


def _p3_wrong_tool_then_correct(obs: List[str], step: int) -> Dict[str, str]:
    # Deliberate failure: step 0 reaches for the calculator on a question
    # the calculator cannot possibly answer (it needs looked-up facts
    # first). This is the "wrong tool for the task" failure mode.
    if step == 0:
        return {"thought": "I need the combined height, so I'll add them directly.",
                "action": "calculator", "input": "Mount Everest height + Eiffel Tower height"}
    if step == 1:
        # obs[-1] is the calculator's ERROR string from the bad call above.
        return {"thought": f"That failed ({obs[-1]}) -- the calculator can't look up facts, only do arithmetic on numbers I already have. I need to search for Mount Everest's height first.",
                "action": "search_docs", "input": "Mount Everest height meters"}
    if step == 2:
        everest_m = _last_number(obs[-1])
        return {"thought": f"Everest is {everest_m:.0f} meters. Now I need the Eiffel Tower's height too.",
                "action": "search_docs", "input": "Eiffel Tower height meters"}
    if step == 3:
        everest_m = _last_number(obs[-2])
        eiffel_m = _first_number(obs[-1])  # Eiffel doc has two numbers (height, year) -- height is first
        return {"thought": f"Now I can add {everest_m:.0f} and {eiffel_m:.0f}.",
                "action": "calculator", "input": f"{everest_m:.0f} + {eiffel_m:.0f}"}
    total = _last_number(obs[-1])
    return {"thought": "I have the combined height.",
            "final_answer": f"Mount Everest and the Eiffel Tower have a combined height of {total:.0f} meters."}


def _p4_looping_no_progress(obs: List[str], step: int) -> Dict[str, str]:
    # Deliberate failure: the knowledge base has no exchange-rate data and
    # there's no exchange-rate tool, but instead of recognizing that after
    # one failed search, the agent tries rephrasing the same query twice
    # more -- three near-identical actions that each return the same
    # "no results" observation. This is the "looping without progress"
    # failure mode.
    queries = [
        "Euro to Yen exchange rate",
        "EUR JPY exchange rate today",
        "current exchange rate Euro Japanese Yen",
    ]
    if step < 3:
        return {"thought": f"Let me search the knowledge base for the exchange rate (attempt {step + 1}).",
                "action": "search_docs", "input": queries[step]}
    return {"thought": "Three searches in a row found nothing, and none of the three tools (calculator, search_docs, get_today) can fetch a live exchange rate.",
            "final_answer": "I can't answer this -- the knowledge base has no exchange-rate data, and none of my tools can fetch live rates."}


def _p5_hallucinated_final_answer(obs: List[str], step: int) -> Dict[str, str]:
    if step == 0:
        return {"thought": "I need the Amazon rainforest's area in square kilometers.",
                "action": "search_docs", "input": "Amazon rainforest area square kilometers"}
    if step == 1:
        sqkm = _last_number(obs[-1])
        return {"thought": f"The area is {sqkm:.0f} sq km. 1 sq km = 0.386 sq miles, so I'll multiply.",
                "action": "calculator", "input": f"{sqkm:.0f} * 0.386"}
    # Deliberate failure: the real Observation (obs[-1]) holds the correct
    # answer, but this step ignores it and states a different, fabricated
    # number instead -- the "hallucinating a tool output it never actually
    # received" failure mode. The correct value is computed here ONLY so
    # the notebook's automated consistency check can prove the mismatch;
    # the scripted Thought/Final Answer text does not use it.
    return {"thought": "I have the converted area.",
            "final_answer": "The Amazon rainforest covers approximately 2,715,500 square miles."}


PROBLEMS: Dict[str, Callable[[List[str], int], Dict[str, str]]] = {
    "How many years ago was Python first released, based on today's date?": _p1_python_release,
    "What is the height of the Eiffel Tower in feet? (1 meter = 3.281 feet)": _p2_eiffel_feet,
    "What is the combined height of Mount Everest and the Eiffel Tower in meters?": _p3_wrong_tool_then_correct,
    "What is the current exchange rate between the Euro and the Japanese Yen?": _p4_looping_no_progress,
    "What is the area of the Amazon rainforest in square miles? (1 sq km = 0.386 sq miles)": _p5_hallucinated_final_answer,
}


def call_llm(question: str, observations: List[str], step: int) -> Dict[str, str]:
    """
    Produce the next agent step (Thought + Action, or Thought + Final Answer).

    Args:
        question: The original question being solved.
        observations: All Observations returned so far, in order (from the
            real tool calls the loop has already executed).
        step: 0-indexed step number within this problem.

    Returns:
        A dict with either {"thought", "action", "input"} for another tool
        call, or {"thought", "final_answer"} to end the loop.

    Raises:
        KeyError: If `question` doesn't match one of the five scripted
            problems below (this mock only knows these five).
    """
    return PROBLEMS[question](observations, step)

Writing llm.py


## 4. The manual ReAct loop

This is the part that would be identical with a real LLM behind it: call the model, check
whether it produced a Final Answer or another Action, execute the Action for real via
`dispatch()`, append the Observation, and loop — capped at `MAX_STEPS` so a genuinely stuck
agent can't run forever.


In [5]:
%%writefile agent.py
"""
agent.py
--------
The manual ReAct loop itself. This is the part of the notebook that stays
identical whether call_llm() is the offline mock in llm.py or a real
OpenAI Chat Completions call: produce a Thought, name an Action, execute it
via dispatch(), feed the Observation back in, repeat until a Final Answer
appears.
"""

from __future__ import annotations

from dataclasses import dataclass, field
from typing import List, Optional

from llm import call_llm
from registry import dispatch

MAX_STEPS = 6


@dataclass
class Step:
    """One Thought/Action/Observation triple (or a final Thought/Answer)."""
    thought: str
    action: Optional[str] = None
    action_input: Optional[str] = None
    observation: Optional[str] = None
    final_answer: Optional[str] = None


@dataclass
class AgentTrace:
    """The full trace of one run: the question and every Step taken."""
    question: str
    steps: List[Step] = field(default_factory=list)

    @property
    def final_answer(self) -> Optional[str]:
        """The Final Answer of the last step, if the agent finished."""
        if self.steps and self.steps[-1].final_answer is not None:
            return self.steps[-1].final_answer
        return None

    def observations(self) -> List[str]:
        """All Observations recorded so far, in order."""
        return [s.observation for s in self.steps if s.observation is not None]


def run_agent(question: str, max_steps: int = MAX_STEPS) -> AgentTrace:
    """
    Run the manual ReAct loop for one question until a Final Answer appears
    or max_steps is reached.

    Args:
        question: The question to solve.
        max_steps: Safety cap on the number of Thought/Action/Observation
            cycles, so a genuinely stuck agent can't loop forever.

    Returns:
        An AgentTrace with every Step taken, in order. If the loop hits
        max_steps without a Final Answer, the trace simply ends without one
        -- callers can check `trace.final_answer is None` to detect this.
    """
    trace = AgentTrace(question=question)

    for step_index in range(max_steps):
        llm_output = call_llm(question, trace.observations(), step_index)

        if "final_answer" in llm_output:
            trace.steps.append(Step(thought=llm_output["thought"], final_answer=llm_output["final_answer"]))
            break

        observation = dispatch(llm_output["action"], llm_output["input"])
        trace.steps.append(Step(
            thought=llm_output["thought"],
            action=llm_output["action"],
            action_input=llm_output["input"],
            observation=observation,
        ))

    return trace


def print_trace(trace: AgentTrace) -> None:
    """Pretty-print a full Thought -> Action -> Observation trace."""
    print(f"Question: {trace.question}")
    print("-" * 70)
    for i, step in enumerate(trace.steps, start=1):
        print(f"Step {i}")
        print(f"  Thought: {step.thought}")
        if step.action is not None:
            print(f"  Action: {step.action}[{step.action_input}]")
            print(f"  Observation: {step.observation}")
        else:
            print(f"  Final Answer: {step.final_answer}")
        print()

Writing agent.py


## 5. Running the agent on 5 multi-step problems

Each problem needs at least two tools used in sequence. Full Thought → Action → Observation
traces are printed for every step of every problem, so the entire reasoning chain is
inspectable.


In [6]:
from agent import run_agent, print_trace
from llm import PROBLEMS

traces = {}
for question in PROBLEMS:
    trace = run_agent(question)
    traces[question] = trace
    print_trace(trace)
    print("=" * 70)
    print()

Question: How many years ago was Python first released, based on today's date?
----------------------------------------------------------------------
Step 1
  Thought: I need to find out when Python was first released.
  Action: search_docs[Python first released year]
  Observation: [doc_python] Python was first released by Guido van Rossum in 1991.

Step 2
  Thought: Python was released in 1991. I need today's date to compute how many years ago that was.
  Action: get_today[]
  Observation: 2026-08-28

Step 3
  Thought: Today is 2026-08-28, so I can calculate 2026 - 1991.
  Action: calculator[2026 - 1991]
  Observation: 35

Step 4
  Thought: I now have the number of years.
  Final Answer: Python was first released in 1991, which is 35 years ago (as of today).


Question: What is the height of the Eiffel Tower in feet? (1 meter = 3.281 feet)
----------------------------------------------------------------------
Step 1
  Thought: I need the Eiffel Tower's height in meters before I can c

## 6. Detecting the failure patterns automatically

Rather than eyeballing the traces and asserting "this one looped," each failure mode gets a
small detector function that inspects the trace programmatically. This is the same discipline
as Day 25's regression diff: a documented finding should be something the code can prove, not
just something the notebook claims in prose.


In [7]:
import re

def detect_wrong_tool(trace):
    """A tool call that errored out is a concrete signal the wrong tool was tried first."""
    return any(s.observation and s.observation.startswith("ERROR:") for s in trace.steps)

def detect_looping(trace):
    """3+ search_docs calls that all came back empty = repeating the same failed approach."""
    empty_searches = sum(
        1 for s in trace.steps
        if s.action == "search_docs" and s.observation == "No relevant documents found."
    )
    return empty_searches >= 3

def detect_hallucination(trace):
    """The Final Answer should contain the number the last successful calculator call
    actually returned. If it doesn't, the model stated a number the tools never produced."""
    calc_results = [
        float(re.findall(r"-?\d+\.?\d*", s.observation)[-1])
        for s in trace.steps
        if s.action == "calculator" and s.observation and not s.observation.startswith("ERROR")
    ]
    if not calc_results or not trace.final_answer:
        return False
    final_numbers = [float(n.replace(",", "")) for n in re.findall(r"[\d,]+\.?\d*", trace.final_answer)]
    last_result = calc_results[-1]
    return not any(abs(n - last_result) < 0.5 for n in final_numbers)

print(f"{'Problem':<58} | Detected failure")
print("-" * 90)
for question, trace in traces.items():
    flags = []
    if detect_wrong_tool(trace):
        flags.append("WRONG TOOL (calculator tried on a fact-lookup question)")
    if detect_looping(trace):
        flags.append("LOOPING (3+ empty searches, no query strategy change)")
    if detect_hallucination(trace):
        flags.append("HALLUCINATION (Final Answer number != last tool Observation)")
    print(f"{question[:58]:<58} | {', '.join(flags) if flags else 'clean'}")

Problem                                                    | Detected failure
------------------------------------------------------------------------------------------
How many years ago was Python first released, based on tod | clean
What is the height of the Eiffel Tower in feet? (1 meter = | clean
What is the combined height of Mount Everest and the Eiffe | WRONG TOOL (calculator tried on a fact-lookup question)
What is the current exchange rate between the Euro and the | LOOPING (3+ empty searches, no query strategy change)
What is the area of the Amazon rainforest in square miles? | HALLUCINATION (Final Answer number != last tool Observation)


### What each detector caught

- **Problem 3 (combined height)** — the agent's *first* move was `calculator("Mount Everest
  height + Eiffel Tower height")`, which is not arithmetic at all — it's two facts it hadn't
  looked up yet. The calculator correctly returned an `ERROR:` Observation instead of
  crashing, and the agent read that error and switched to `search_docs` on the next step. This
  is the **wrong tool selected first** failure mode — caught here because it *recovered*, but
  a less careful agent could easily have kept retrying the same broken calculator call instead.
- **Problem 4 (exchange rate)** — none of the three tools can answer this question at all (no
  exchange-rate data exists in the knowledge base, and there's no live-rates tool). Instead of
  recognizing that after one failed search, the agent tried two more near-identical rephrasings
  that were always going to return the same "No relevant documents found." This is **looping
  without progress** — three actions, zero new information gained between them.
- **Problem 5 (Amazon rainforest area)** — the calculator actually returned the correct value
  (`2123000.0` sq miles), but the Final Answer states **2,715,500** — a number that appears
  nowhere in the trace. This is a **hallucinated Observation**: the model asserted a plausible-
  sounding number instead of the one its own tool call actually produced. This is the most
  dangerous failure mode of the three, because unlike the other two, nothing in the trace
  *looks* broken — the Thought and Action lines are perfectly reasonable, and only checking
  the Final Answer against the real Observation catches it.


## 7. What makes an agent reliable vs. unpredictable

Grounded specifically in the three failures observed above, not in the two clean runs:

**1. Reliability tracks how tightly a step is bound to the tool's actual return value, not
how good the reasoning text sounds.** Problems 1, 2, and the second half of Problem 3 all read
a number *out of* the previous Observation with a regex before using it — the model's next
action was mechanically derived from what the tool actually said. Problem 5's Final Answer, by
contrast, was generated independently of the calculator's return value — the Thought
("I have the converted area") sounds identical to a correct step, but nothing enforced that the
number in the answer had to match the number in the Observation two lines above it. **A
reliable agent step is one where the reasoning text is causally downstream of the tool output,
not just stylistically consistent with it.**

**2. An agent is only as good as its ability to recognize "this approach isn't working" —
and that recognition has to be about the *tool result*, not the *attempt count*.** Problem 3
self-corrected after exactly one error because the calculator's `ERROR:` string was
informative enough to change strategy. Problem 4 didn't self-correct after three empty
searches because "No relevant documents found." carries no information about *why* — the
agent had no way to distinguish "try a different phrasing" from "this data doesn't exist and
no phrasing will help." **Tool observations that explain a failure (Problem 3's calculator
error) are recoverable; tool observations that only report absence (Problem 4's search) invite
looping**, because the model has nothing to reason about except "try again."

**3. The wrong-tool and looping failures are visible in the trace; the hallucination failure
is not.** Anyone reading Problem 3's or Problem 4's trace start-to-finish would immediately
spot the error message or the three repeated searches. Problem 5's trace reads as completely
normal end to end — the failure only exists in the *gap* between the last Observation and the
Final Answer, which is exactly the part of a ReAct trace people are least likely to
double-check once the earlier steps look fine. **This is the actual argument for keeping the
full trace, not just the final answer: the failure that matters most is the one that looks
the most like success.**

**Bottom line:** an agent is reliable in proportion to how mechanically its next step follows
from the last tool's real output, and unpredictable in exactly the gaps where that link is
implicit instead of enforced — whether that gap is "the model didn't check *why* a search came
back empty" (Problem 4) or "nothing forced the final number to match the calculator's return
value" (Problem 5). A framework that hides the loop hides those gaps too.
